In [5]:
import os
import numpy as np
from scipy.io import loadmat
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ================= 配置 =================
rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
rcParams['axes.unicode_minus'] = False

DATA_DIR = r'.\\data\\WT-行星齿轮数据集'
FILE_MAPPING = {
    'B1_35.MAT': 'Broken',
    'N1_35.MAT': 'Healthy',
    'M1_35.MAT': 'Missing_tooth',
    'R1_35.MAT': 'Root_crack',
    'W1_35.MAT': 'Wear'
}
WINDOW_SIZE = 2048
STRIDE = 2048


# ================= 归一化工具函数（Z-score）=================
def standardize_signal(x: np.ndarray) -> tuple:
    """Z-score 标准化: (x - mean) / std"""
    x_mean = x.mean()
    x_std = x.std()
    eps = 1e-8
    if x_std < eps:
        x_std = eps
        standardized_x = np.zeros_like(x)
    else:
        standardized_x = (x - x_mean) / x_std
    return standardized_x, {'mean': x_mean, 'std': x_std}


def destandardize_signal(x_norm: np.ndarray, mean: float,
                         std: float) -> np.ndarray:
    """反标准化: x * std + mean"""
    return x_norm * std + mean


print("🔄 正在流式加载数据：先 Z-score 标准化整条信号，再切分窗口...")

X_segments = []
y_labels = []

# 保存每个文件的标准化参数（用于后续可能的反标准化）
file_stats = {}

for file_name, fault_type in FILE_MAPPING.items():
    file_path = os.path.join(DATA_DIR, file_name)
    if not os.path.exists(file_path):
        print(f"⚠️ 跳过缺失文件: {file_name}")
        continue

    try:
        # 加载 .mat 数据
        mat_data = loadmat(file_path)
        raw_data = None
        for k, v in mat_data.items():
            if not k.startswith('__') and isinstance(
                    v, np.ndarray) and v.ndim == 2 and v.shape[1] >= 1:
                raw_data = v[:, :1]  # 只取第1列（注意：原代码写的是[:1]，不是前两列）
                break
        if raw_data is None:
            print(f"❌ 跳过无效文件: {file_name}")
            continue

        n = len(raw_data)
        if n < WINDOW_SIZE:
            print(f"⚠️ {file_name} 数据太短 ({n} < {WINDOW_SIZE})，跳过")
            continue

        # === 先对整条信号做 Z-score 标准化 ===
        standardized_raw, stats = standardize_signal(raw_data)
        file_stats[file_name] = stats  # 保存 mean/std

        # === 再切分窗口 ===
        count = 0
        for i in range(0, n - WINDOW_SIZE + 1, STRIDE):
            segment = standardized_raw[i:i + WINDOW_SIZE]  # [L, 1]
            X_segments.append(segment.copy())
            y_labels.append(fault_type)
            count += 1

        print(f"✅ {file_name} ({fault_type}): 标准化后切分 {count} 个窗口")

    except Exception as e:
        print(f"❌ 处理 {file_name} 出错: {e}")

if not X_segments:
    raise RuntimeError("未生成任何有效样本！")

# 转为 NumPy 数组
X_all = np.stack(X_segments, axis=0)  # [N, L, C]
y_all = np.array(y_labels)

print(f"\n✅ 总样本数: {len(X_all)}, 形状: {X_all.shape}")

🔄 正在流式加载数据：先 Z-score 标准化整条信号，再切分窗口...
✅ B1_35.MAT (Broken): 标准化后切分 7064 个窗口
✅ N1_35.MAT (Healthy): 标准化后切分 7080 个窗口
✅ M1_35.MAT (Missing_tooth): 标准化后切分 7096 个窗口
✅ R1_35.MAT (Root_crack): 标准化后切分 7120 个窗口
✅ W1_35.MAT (Wear): 标准化后切分 7064 个窗口

✅ 总样本数: 35424, 形状: (35424, 2048, 1)


In [6]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# ================================
# 🎯 保留原始多类标签（不转二分类！）
# ================================

# 假设 y_all 是字符串标签，例如: ['Healthy', 'InnerRace', 'Ball', 'OuterRace', ...]
# X_all: [N, L, 1] 或 [N, L] 的 numpy array

# 构建类别映射（字符串 ↔ 整数）
unique_labels = np.unique(y_all)
label_to_int = {label: idx for idx, label in enumerate(unique_labels)}
int_to_label = {idx: label for label, idx in label_to_int.items()}
num_classes = len(unique_labels)

print(f"🏷️ 多分类标签映射 (共 {num_classes} 类):")
for label, idx in label_to_int.items():
    print(f"   '{label}' → {idx}")

# 将字符串标签转为整数标签
y_all_int = np.array([label_to_int[label] for label in y_all])

# 提取正常数据（用于预训练）—— 仍用原始字符串判断
healthy_mask = (y_all == 'Healthy')
X_healthy = X_all[healthy_mask]
print(f"🟢 正常样本数（用于预训练）: {len(X_healthy)}")

# ================================
# 🔀 划分数据集（按多类标签分层）
# ================================

X_temp, X_test, y_temp_int, y_test_int = train_test_split(
    X_all,
    y_all_int,
    test_size=0.2,
    random_state=42,
    stratify=y_all_int  # 👈 按多类分层，确保每类在各集合中都有
)

X_train, X_val, y_train_int, y_val_int = train_test_split(X_temp,
                                                          y_temp_int,
                                                          test_size=0.25,
                                                          random_state=42,
                                                          stratify=y_temp_int)

# ================================
# 📦 Dataset 定义（支持多分类）
# ================================


class PretrainDataset(Dataset):

    def __init__(self, data_np):
        # 假设输入是 [N, L, 1] 或 [N, L]
        if data_np.ndim == 2:
            data_np = data_np[:, :, np.newaxis]  # 转为 [N, L, 1]
        self.data = torch.FloatTensor(data_np).permute(0, 2, 1)  # [N, 1, L]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


class FinetuneDataset(Dataset):

    def __init__(self, data_np, labels_int):
        if data_np.ndim == 2:
            data_np = data_np[:, :, np.newaxis]
        self.data = torch.FloatTensor(data_np).permute(0, 2, 1)
        self.labels = torch.LongTensor(labels_int)  # 已是整数标签

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# ================================
# 🚀 创建 DataLoader
# ================================

batch_size = 64

pretrain_loader = DataLoader(PretrainDataset(X_healthy),
                             batch_size=batch_size,
                             shuffle=True)

train_loader = DataLoader(FinetuneDataset(X_train, y_train_int),
                          batch_size=batch_size,
                          shuffle=True)

val_loader = DataLoader(FinetuneDataset(X_val, y_val_int),
                        batch_size=batch_size,
                        shuffle=False)

test_loader = DataLoader(FinetuneDataset(X_test, y_test_int),
                         batch_size=batch_size,
                         shuffle=False)

# ================================
# 📊 打印统计信息
# ================================

from collections import Counter


def count_labels(y_int, int_to_label):
    counts = Counter(y_int)
    return {int_to_label[idx]: cnt for idx, cnt in sorted(counts.items())}


print(f"\n✅ 多分类数据管道构建完成！(共 {num_classes} 类)")
print(
    f"   - 训练集: {len(y_train_int)} 样本 → {count_labels(y_train_int, int_to_label)}"
)
print(
    f"   - 验证集: {len(y_val_int)} 样本 → {count_labels(y_val_int, int_to_label)}")
print(
    f"   - 测试集: {len(y_test_int)} 样本 → {count_labels(y_test_int, int_to_label)}"
)

🏷️ 多分类标签映射 (共 5 类):
   'Broken' → 0
   'Healthy' → 1
   'Missing_tooth' → 2
   'Root_crack' → 3
   'Wear' → 4
🟢 正常样本数（用于预训练）: 7080

✅ 多分类数据管道构建完成！(共 5 类)
   - 训练集: 21254 样本 → {np.str_('Broken'): 4238, np.str_('Healthy'): 4248, np.str_('Missing_tooth'): 4258, np.str_('Root_crack'): 4272, np.str_('Wear'): 4238}
   - 验证集: 7085 样本 → {np.str_('Broken'): 1413, np.str_('Healthy'): 1416, np.str_('Missing_tooth'): 1419, np.str_('Root_crack'): 1424, np.str_('Wear'): 1413}
   - 测试集: 7085 样本 → {np.str_('Broken'): 1413, np.str_('Healthy'): 1416, np.str_('Missing_tooth'): 1419, np.str_('Root_crack'): 1424, np.str_('Wear'): 1413}


In [7]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# 假设你已有：
# X_all: [N, L] 或 [N, L, 1]
# y_all: 原始字符串标签，如 ['Healthy', 'InnerRace', 'Ball', ...]

# ================================
# 🎯 步骤 1: 按原始多类标签划分数据集（分层）
# ================================

unique_labels = np.unique(y_all)
label_to_int = {label: i for i, label in enumerate(unique_labels)}
y_all_int = np.array([label_to_int[y] for y in y_all])

X_temp, X_test, y_temp_str, y_test_str = train_test_split(X_all,
                                                          y_all,
                                                          test_size=0.2,
                                                          random_state=42,
                                                          stratify=y_all_int)

X_train, X_val, y_train_str, y_val_str = train_test_split(
    X_temp,
    y_temp_str,
    test_size=0.25,
    random_state=42,
    stratify=[label_to_int[y] for y in y_temp_str])

print("✅ 数据集划分完成（按原始故障类型分层）")

# ================================
# 🎯 步骤 2: 小样本采样函数（仅用于 train/val）
# ================================


def sample_fewshot_binary(X, y_str, n_per_class=100, random_seed=42):
    rng = np.random.RandomState(random_seed)
    unique = np.unique(y_str)
    X_list, y_bin_list, y_str_list = [], [], []

    for label in unique:
        mask = (y_str == label)
        X_label = X[mask]
        y_label = y_str[mask]
        n_avail = len(X_label)
        n_take = min(n_per_class, n_avail)

        if n_take < n_avail:
            idx = rng.choice(n_avail, n_take, replace=False)
            X_label = X_label[idx]
            y_label = y_label[idx]

        X_list.append(X_label)
        y_str_list.append(y_label)
        bin_val = 0 if label == 'Healthy' else 1
        y_bin_list.append(np.full(len(X_label), bin_val))

    X_out = np.concatenate(X_list, axis=0)
    y_bin_out = np.concatenate(y_bin_list, axis=0)
    y_str_out = np.concatenate(y_str_list, axis=0)

    shuffle_idx = rng.permutation(len(X_out))
    return X_out[shuffle_idx], y_bin_out[shuffle_idx], y_str_out[shuffle_idx]


# ================================
# 🚀 应用小样本采样（仅训练集和验证集）
# ================================

n_per_class = 100

X_train_fs, y_train_bin, y_train_str_fs = sample_fewshot_binary(
    X_train, y_train_str, n_per_class=n_per_class, random_seed=42)

X_val_fs, y_val_bin, y_val_str_fs = sample_fewshot_binary(
    X_val, y_val_str, n_per_class=n_per_class, random_seed=123)


# ✅ 测试集：不采样！保留全部样本，仅转为二分类
def to_binary_labels_str(y_str):
    return np.where(y_str == 'Healthy', 0, 1).astype(int)


X_test_fs = X_test  # 全量测试集
y_test_bin = to_binary_labels_str(y_test_str)
y_test_str_fs = y_test_str  # 用于分析

# ================================
# 📦 Dataset 和 DataLoader
# ================================


class FinetuneDataset(Dataset):

    def __init__(self, data_np, labels):
        if data_np.ndim == 2:
            data_np = data_np[:, :, np.newaxis]
        self.data = torch.FloatTensor(data_np).permute(0, 2, 1)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


batch_size = 64

train_loader = DataLoader(FinetuneDataset(X_train_fs, y_train_bin),
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(FinetuneDataset(X_val_fs, y_val_bin),
                        batch_size=batch_size,
                        shuffle=False)
test_loader = DataLoader(FinetuneDataset(X_test_fs, y_test_bin),
                         batch_size=batch_size,
                         shuffle=False)

# ================================
# 📊 打印统计信息
# ================================


def print_fault_distribution(name, y_str):
    counts = Counter(y_str)
    healthy = counts.get('Healthy', 0)
    faulty_total = sum(v for k, v in counts.items() if k != 'Healthy')
    fault_types = {k: v for k, v in counts.items() if k != 'Healthy'}
    print(f"\n{name} 集:")
    print(f"  总样本: {len(y_str)} | Healthy: {healthy}, Faulty: {faulty_total}")
    print(f"  故障子类分布: {dict(sorted(fault_types.items()))}")


print("\n" + "=" * 60)
print(f"✅ 小样本二分类数据集构建完成！(Train/Val 每原始类 ≤{n_per_class} 样本，Test 全量)")
print("=" * 60)
print_fault_distribution("训练", y_train_str_fs)
print_fault_distribution("验证", y_val_str_fs)
print_fault_distribution("测试", y_test_str_fs)  # 显示完整测试分布

print(f"\n📊 二分类统计:")
print(
    f"   - 训练集: {len(y_train_bin)} 样本 (Healthy: {(y_train_bin==0).sum()}, Faulty: {(y_train_bin==1).sum()})"
)
print(
    f"   - 验证集: {len(y_val_bin)} 样本 (Healthy: {(y_val_bin==0).sum()}, Faulty: {(y_val_bin==1).sum()})"
)
print(
    f"   - 测试集: {len(y_test_bin)} 样本 (Healthy: {(y_test_bin==0).sum()}, Faulty: {(y_test_bin==1).sum()}) [全量]"
)

✅ 数据集划分完成（按原始故障类型分层）

✅ 小样本二分类数据集构建完成！(Train/Val 每原始类 ≤100 样本，Test 全量)

训练 集:
  总样本: 500 | Healthy: 100, Faulty: 400
  故障子类分布: {np.str_('Broken'): 100, np.str_('Missing_tooth'): 100, np.str_('Root_crack'): 100, np.str_('Wear'): 100}

验证 集:
  总样本: 500 | Healthy: 100, Faulty: 400
  故障子类分布: {np.str_('Broken'): 100, np.str_('Missing_tooth'): 100, np.str_('Root_crack'): 100, np.str_('Wear'): 100}

测试 集:
  总样本: 7085 | Healthy: 1416, Faulty: 5669
  故障子类分布: {np.str_('Broken'): 1413, np.str_('Missing_tooth'): 1419, np.str_('Root_crack'): 1424, np.str_('Wear'): 1413}

📊 二分类统计:
   - 训练集: 500 样本 (Healthy: 100, Faulty: 400)
   - 验证集: 500 样本 (Healthy: 100, Faulty: 400)
   - 测试集: 7085 样本 (Healthy: 1416, Faulty: 5669) [全量]


In [8]:
sample_x = next(iter(val_loader))[1]
sample_x.shape

torch.Size([64])

In [9]:
sample_x

tensor([1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1])

In [86]:
import torch
import torch.nn as nn
import math
from typing import Tuple
import random


class BlockMaskedEncoder(nn.Module):
    """
    Encoder-only 单变量时间序列掩码重建模型（MLM 风格）
    
    特点：
      - 使用你指定的 block-wise masking 策略（自适应 block 长度）
      - 全序列输入 encoder（masked 位置用 [MASK] token）
      - 无 decoder，纯 Encoder-only
      - 支持预训练、重建、特征提取、参数统计
    """

    def __init__(self,
                 signal_length: int,
                 patch_size: int = 64,
                 embed_dim: int = 128,
                 encoder_depth: int = 4,
                 num_heads: int = 8,
                 mask_ratio: float = 0.75,
                 dropout: float = 0.1):
        super().__init__()
        self.signal_length = signal_length
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio
        self.num_patches = math.ceil(signal_length / patch_size)

        # === Patch Embedding ===
        self.patch_embed = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            padding=0
        )
        self.norm = nn.LayerNorm(embed_dim)

        # === [MASK] token ===
        self.mask_token = nn.Parameter(torch.zeros(embed_dim))
        nn.init.normal_(self.mask_token, std=0.02)

        # === Positional Embedding ===
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.normal_(self.pos_embed, std=0.02)

        # === Encoder ===
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 6,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=encoder_depth)
        self.encoder_norm = nn.LayerNorm(embed_dim)

        # === Reconstruction Head ===
        self.recon_head = nn.Sequential(nn.Linear(embed_dim, embed_dim),
                                        nn.GELU(), nn.LayerNorm(embed_dim),
                                        nn.Linear(embed_dim, patch_size))
        self._init_weights()


    def _init_weights(self):
        # ... 其他初始化 ...

        # 初始化 recon_head
        for module in self.recon_head:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.LayerNorm):
                nn.init.constant_(module.bias, 0)
                nn.init.constant_(module.weight, 1.0)

    def _pad_signal(self, x: torch.Tensor) -> torch.Tensor:
        L = x.size(-1)
        if L % self.patch_size != 0:
            pad_len = self.patch_size - (L % self.patch_size)
            x = nn.functional.pad(x, (0, pad_len))
        return x


    def _generate_block_mask(self, B: int, N: int) -> torch.Tensor:
        """
        高效生成至少 3 个不连续的连续 mask 块（块内连续，块间尽量不邻）
        向量化为主，仅轻量后处理，速度比逐样本快 5~10 倍
        """
        device = next(self.parameters()).device
        min_blocks = 3
        max_blocks = max(min_blocks, N // 4)  # 最多 N//4 个块
        max_block_len = max(1, N // 8)
        min_block_len = 1

        # 步骤1: 随机决定每个样本要多少个块（至少3）
        num_blocks_per_sample = torch.randint(min_blocks,
                                            max_blocks + 1, (B, ),
                                            device=device)  # [B]

        # 步骤2: 为每个样本生成足够多的候选起始位置和长度（批量）
        max_num_blocks = num_blocks_per_sample.max().item()
        starts = torch.randint(0, N, (B, max_num_blocks), device=device)
        lengths = torch.randint(min_block_len,
                                max_block_len + 1, (B, max_num_blocks),
                                device=device)
        ends = torch.clamp(starts + lengths, max=N)

        # 步骤3: 构建 mask 张量（初始全 False）
        mask = torch.zeros(B, N, dtype=torch.bool, device=device)

        # 步骤4: 向量化填充每个块（允许重叠/相邻，后续修正）
        # 使用 scatter_add 模拟区间赋值（技巧：用 cumsum）
        for i in range(max_num_blocks):
            # 获取当前 batch 的有效块（避免越界）
            valid = i < num_blocks_per_sample  # [B]
            if not valid.any():
                break
            s = starts[valid, i]  # [B_valid]
            e = ends[valid, i]  # [B_valid]
            idx = torch.arange(N, device=device).unsqueeze(0)  # [1, N]
            # 广播比较：[B_valid, N]
            block_mask = (idx >= s.unsqueeze(1)) & (idx < e.unsqueeze(1))
            mask[valid] |= block_mask

        # 步骤5: 确保总 mask 比例接近目标
        target_masked = int(N * self.mask_ratio + 0.5)
        current_sum = mask.sum(dim=1)  # [B]

        # 如果某些样本 mask 太少，随机补点（不破坏块结构太多）
        for b in range(B):
            need = target_masked - current_sum[b].item()
            if need > 0:
                unmasked = torch.where(~mask[b])[0]
                if len(unmasked) > 0:
                    take = min(need, len(unmasked))
                    idx = unmasked[torch.randperm(len(unmasked),
                                                device=device)[:take]]
                    mask[b, idx] = True
            elif need < 0:
                # 如果 mask 太多，随机取消一些（优先取消孤立点）
                masked = torch.where(mask[b])[0]
                drop = min(-need, len(masked))
                idx = masked[torch.randperm(len(masked), device=device)[:drop]]
                mask[b, idx] = False

        # ✅ 可选：强制断开长连续段（提升“不连续性”）
        # 对每个样本，检测连续段，若某段太长则从中挖洞
        # （此处省略，因已通过多块+随机起始降低连续概率）

        return mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, L = x.shape
        assert C == 1 and L == self.signal_length

        x_padded = self._pad_signal(x)  # [B, 1, L_pad]
        patches = self.patch_embed(x_padded).transpose(1, 2)  # [B, N, D]
        patches = self.norm(patches)
        N = patches.size(1)

        # ✅ 新：直接生成 block-wise mask
        mask = self._generate_block_mask(B, N)  # [B, N], bool

        # ✅ 安全可微分替换（关键！避免 in-place 赋值）
        x_masked = torch.where(mask.unsqueeze(-1), self.mask_token, patches)
        x_masked = x_masked + self.pos_embed

        encoded = self.encoder_norm(self.encoder(x_masked))
        pred = self.recon_head(encoded)
        target = x_padded.unfold(2, self.patch_size, self.patch_size).squeeze(1)

        loss = ((pred - target) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / (mask.sum() + 1e-8)
        return loss
    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # === 自动修复输入维度 ===
        if x.ndim == 1:
            x = x.unsqueeze(0).unsqueeze(0)  # [T] → [1, 1, T]
        elif x.ndim == 2:
            if x.shape[0] == 1 and x.shape[1] == self.signal_length:
                x = x.unsqueeze(1)  # [1, T] → [1, 1, T]
            else:
                x = x.unsqueeze(0)  # [C=1, T] → [1, 1, T] （假设单通道）
        # 现在 x 应为 [B, 1, L]
        B,C, L = x.shape

        assert C == 1 and L == self.signal_length

        x_orig = x.squeeze(1)  # [B, L]
        x_padded = self._pad_signal(x)  # [B, 1, L_pad]
        L_pad = x_padded.size(-1)

        patches = self.patch_embed(x_padded).transpose(1, 2)
        patches = self.norm(patches)
        N = patches.size(1)

        # ✅ 新：直接生成 block-wise mask
        mask = self._generate_block_mask(B, N)  # [B, N], bool

        # ✅ 安全可微分替换（关键！避免 in-place 赋值）
        x_masked = torch.where(mask.unsqueeze(-1), self.mask_token, patches)
        x_masked = x_masked + self.pos_embed


        encoded = self.encoder_norm(self.encoder(x_masked))
        pred = self.recon_head(encoded)
        x_recon_full = pred.reshape(B, N * self.patch_size)

        mask_time = mask.unsqueeze(-1).repeat(1, 1, self.patch_size).reshape(B, L_pad)
        x_padded_flat = x_padded.squeeze(1)
        x_recon_hybrid = torch.where(mask_time, x_recon_full, x_padded_flat)

        return x_orig, x_recon_hybrid[:, :L], mask_time[:, :L]

    @torch.no_grad()
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        B, C, L = x.shape
        assert C == 1 and L == self.signal_length
        x_padded = self._pad_signal(x)
        patches = self.patch_embed(x_padded).transpose(1, 2)
        patches = self.norm(patches) + self.pos_embed
        encoded = self.encoder_norm(self.encoder(patches))
        return encoded.mean(dim=1)  # [B, D]


    def check_dims(self):
        """打印模型各模块参数量，用于维度检查"""

        def count_params(module):
            return sum(p.numel() for p in module.parameters() if p.requires_grad)

        print("=== 模型参数量统计 ===")
        print(f"Patch Embedding (Conv1d): {count_params(self.patch_embed):,}")
        print(f"LayerNorm after patch embed: {count_params(self.norm):,}")
        print(f"[MASK] token: {self.mask_token.numel():,}")
        print(f"Positional Embedding: {self.pos_embed.numel():,}")
        print(f"Encoder Transformer: {count_params(self.encoder):,}")
        print(f"Encoder LayerNorm: {count_params(self.encoder_norm):,}")
        print(f"Reconstruction Head: {count_params(self.recon_head):,}")
        total = sum(p.numel() for p in self.parameters())
        print(f"总计: {total:,} 个参数")

In [87]:
model = BlockMaskedEncoder(signal_length=2048)
model.check_dims()

=== 模型参数量统计 ===
Patch Embedding (Conv1d): 8,320
LayerNorm after patch embed: 256
[MASK] token: 128
Positional Embedding: 4,096
Encoder Transformer: 1,056,256
Encoder LayerNorm: 256
Reconstruction Head: 25,024
总计: 1,094,336 个参数


In [106]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# 配置（单通道）
# ----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "./checkpoints_mae"
VIS_DIR = "./mae_reconstructions"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

CHANNEL_IDX = 0
SIGNAL_LENGTH = X_healthy.shape[1]


# 训练参数
EPOCHS = 1000
LR = 1e-5
WEIGHT_DECAY = 0.05
SAVE_EVERY = 5
PATIENCE = 60
EARLY_STOPPING_DELTA = 1e-5

print(f"✅ 使用通道 {CHANNEL_IDX}")
print(f"📊 信号长度: {SIGNAL_LENGTH}")
print(f"📁 Checkpoint: {CHECKPOINT_DIR}")
print(f"🖼️  可视化: {VIS_DIR}")

# ----------------------------
# 初始化模型与优化器
# ----------------------------
model = BlockMaskedEncoder(signal_length=2048, mask_ratio=0.75).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

if hasattr(model, 'print_summary'):
    model.print_summary()

# ----------------------------
# 🚀 简洁加载：仅加载权重，重置所有训练状态
# ----------------------------
name = f"mae_ch{CHANNEL_IDX}_3184_best.pth"
CKPT_PATH = os.path.join(CHECKPOINT_DIR, name)

# 初始训练状态（全新开始！）
start_epoch = 0
best_loss = float('inf')
epochs_no_improve = 0
early_stop = False

# 尝试加载预训练权重（仅用于初始化模型）
if os.path.exists(CKPT_PATH):
    print(f"🔄 加载预训练权重: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    print("   → 模型权重已加载，训练状态已重置（全新早停）")
else:
    print("🆕 未找到预训练模型，从头开始训练。")

# ----------------------------
# 训练循环
# ----------------------------
train_start_time = datetime.now()
for epoch in range(start_epoch, EPOCHS):
    if early_stop:
        print("🛑 触发早停！训练结束。")
        break

    model.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for x in pbar:
        x = x.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast('cuda'):
                loss = model(x)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         grad_norm = param.grad.norm().item()
        #         print(f"{name}: grad_norm = {grad_norm:.6f}")
        #     else:
        #         print(f"{name}: NO GRAD!")
        else:
            loss = model(x)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(Loss=f"{loss.item():.4f}")

    avg_loss = total_loss / num_batches

    # ----------------------------
    # 早停 & 保存最佳模型
    # ----------------------------
    is_best = False
    if avg_loss < best_loss - EARLY_STOPPING_DELTA:
        best_loss = avg_loss
        epochs_no_improve = 0
        is_best = True
    else:
        epochs_no_improve += 1

    # --- 🎯 关键修改：使用你想要的紧凑日志格式 ---
    elapsed = datetime.now() - train_start_time
    total_sec = int(elapsed.total_seconds())
    time_str = f"{total_sec // 60}m {total_sec % 60}s"
    if epoch % 10 == 0:
        # 主日志（两行）
        print(f"Train_Loss: {avg_loss:.6f} | Best_Loss:  {best_loss:.6f}")
        print(
            f"Time: {time_str:<9} | No_Improve: {epochs_no_improve} | Epoch {epoch+1}/{EPOCHS}"
        )

    # 如果是新最佳，额外提示
    if is_best:
        torch.save({'model': model.state_dict()}, CKPT_PATH)
        print(f"   ⭐ 发现更佳模型！已保存 (Loss: {best_loss:.6f})")

    # 早停检查
    if epochs_no_improve >= PATIENCE:
        early_stop = True
        print(f"\n🛑 早停触发！最终最佳 Loss: {best_loss:.6f}")
    # ----------------------------
    # 可视化重建（用 axvspan 画 mask 区域为透明色块）
    # ----------------------------
    if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            # 随机取一个样本
            sample_x = next(iter(pretrain_loader))[0][:1].to(
                DEVICE)  # shape: [1, 1, L]

            x_orig, x_recon, mask_time = model.reconstruct(sample_x)

            x_orig = x_orig.cpu().numpy().flatten()
            x_recon = x_recon.cpu().numpy().flatten()
            mask_time = mask_time.cpu().numpy().flatten()

            plt.figure(figsize=(12, 4))
            plt.plot(x_orig, label='Original', alpha=0.7, linewidth=1)

            masked_recon = np.where(mask_time, x_recon, np.nan)
            plt.plot(masked_recon,
                     label='Reconstructed in Mask',
                     alpha=0.9,
                     linewidth=1,
                     color='orange')

            # 绘制 mask 区域
            if mask_time.any():
                padded = np.concatenate([[False], mask_time, [False]])
                diff = np.diff(padded.astype(int))
                starts = np.where(diff == 1)[0]
                ends = np.where(diff == -1)[0]
                for i, (start, end) in enumerate(zip(starts, ends)):
                    plt.axvspan(start,
                                end,
                                color='lightcoral',
                                alpha=0.4,
                                label='Masked Region' if i == 0 else "")

            plt.title(
                f'Channel {CHANNEL_IDX} | Epoch {epoch+1} | Loss: {avg_loss:.4f}'
            )
            plt.xlabel('Time Step')
            plt.ylabel('Amplitude')
            plt.legend()
            plt.tight_layout()

            suffix = "_best" if is_best else ""
            vis_path = os.path.join(VIS_DIR,
                                    f"{name}_epoch_{epoch+1:03d}{suffix}.png")
            plt.savefig(vis_path, dpi=150)
            plt.close()
            print(f"   🖼️  重建图已保存: {vis_path}")

# ----------------------------
# 最终提示
# ----------------------------
print("\n🎉 单通道 MAE 训练完成！")
print(f"✅ 最佳模型: {CKPT_PATH}")
print(f"📊 最佳损失: {best_loss:.6f}")
print(f"🖼️  重建可视化: {VIS_DIR}/")

✅ 使用通道 0
📊 信号长度: 2048
📁 Checkpoint: ./checkpoints_mae
🖼️  可视化: ./mae_reconstructions
🔄 加载预训练权重: ./checkpoints_mae\mae_ch0_3184_best.pth
   → 模型权重已加载，训练状态已重置（全新早停）


Train_Loss: 0.426438 | Best_Loss:  0.426438
Time: 0m 3s     | No_Improve: 0 | Epoch 1/1000
   ⭐ 发现更佳模型！已保存 (Loss: 0.426438)


   ⭐ 发现更佳模型！已保存 (Loss: 0.425542)


   ⭐ 发现更佳模型！已保存 (Loss: 0.424978)


   ⭐ 发现更佳模型！已保存 (Loss: 0.423110)


...

   🖼️  重建图已保存: ./mae_reconstructions\mae_ch0_3184_best.pth_epoch_175.png


   🖼️  重建图已保存: ./mae_reconstructions\mae_ch0_3184_best.pth_epoch_180.png


Train_Loss: 0.421647 | Best_Loss:  0.418085
Time: 11m 40s   | No_Improve: 55 | Epoch 181/1000


   🖼️  重建图已保存: ./mae_reconstructions\mae_ch0_3184_best.pth_epoch_185.png



🛑 早停触发！最终最佳 Loss: 0.418085
🛑 触发早停！训练结束。

🎉 单通道 MAE 训练完成！
✅ 最佳模型: ./checkpoints_mae\mae_ch0_3184_best.pth
📊 最佳损失: 0.418085
🖼️  重建可视化: ./mae_reconstructions/


In [108]:
import torch
import torch.nn as nn
from collections import OrderedDict


class MAEClassifier(nn.Module):

    def __init__(self,
                 encoder: BlockMaskedEncoder,
                 num_classes: int = 2,
                 freeze_encoder: bool = True):
        super().__init__()
        self.encoder = encoder
        self.num_classes = num_classes

        # 冻结编码器（Linear Probing）或允许微调（Full Fine-tuning）
        for param in self.encoder.parameters():
            param.requires_grad = not freeze_encoder

        # 分类头（接在全局特征后）
        self.classifier = nn.Sequential(
            nn.Linear(encoder.patch_embed.out_channels, 512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes))

    def forward(self, x):
        # x: [B, 1, L]
        features = self.encoder.extract_features(x)  # [B, D]
        logits = self.classifier(features)  # [B, num_classes]
        return logits
# ========== 配置 ==========
num_epochs = 100
freeze_encoder = True  # 👈 先定义！True 表示 Linear Probing，False 表示 Full Fine-tuning
lr = 1e-3 if freeze_encoder else 5e-5  # 微调时学习率要小
weight_decay = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 假设 model 是你已加载的预训练 SignalReconstructionBlock 实例
# 如果还没加载，请确保下面这行有定义：
SIGNAL_LENGTH = X_healthy.shape[1]

MODEL_CFG = dict(
    signal_length=SIGNAL_LENGTH,

)
# 正确加载预训练权重
checkpoint = torch.load("checkpoints_mae/mae_ch0_3184_best.pth",
                        map_location=device)
# 或者用反斜杠（但建议用正斜杠或 os.path.join）
# checkpoint = torch.load(r"checkpoints_mae\mae_ch0_best.pth", map_location=device)

# 提取模型权重（关键！）
state_dict = checkpoint["model"]  # 👈 假设保存时用了 'model' 作为键

# 加载到你的模型
model.load_state_dict(state_dict, strict=True)



classifier_model = MAEClassifier(encoder=model,

                                 freeze_encoder=freeze_encoder).to(device)

optimizer = torch.optim.AdamW(classifier_model.parameters(),
                              lr=lr,
                              weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                       T_max=num_epochs)


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


num_trainable = count_trainable_params(classifier_model)
print(f"🔍 实验 A（预训练+冻结）可训练参数量: {num_trainable:,} (约 {num_trainable/1e3:.1f}K)")

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(torch.cat(all_labels), torch.cat(all_preds))
    return avg_loss, acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(torch.cat(all_labels), torch.cat(all_preds))
    return avg_loss, acc


# ========== 早停 & 内存缓存配置 ==========
PATIENCE = 20
EARLY_STOP_DELTA = 1e-4
best_val_acc = 0.0
epochs_no_improve = 0
early_stop = False
best_model_state = None  # 👈 用于在内存中保存最佳模型参数

for epoch in range(num_epochs):
    if early_stop:
        print("🛑 触发早停！训练提前结束。")
        break

    train_loss, train_acc = train_epoch(classifier_model, train_loader,
                                        criterion, optimizer, device)
    val_loss, val_acc = validate(classifier_model, val_loader, criterion,
                                 device)
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # --- 🛑 早停 + 内存缓存最佳模型 ---
    if val_acc > best_val_acc + EARLY_STOP_DELTA:
        best_val_acc = val_acc
        epochs_no_improve = 0
        # 💾 不保存到磁盘，而是深拷贝 state_dict 到内存
        best_model_state = {
            k: v.cpu()
            for k, v in classifier_model.state_dict().items()
        }  # 转为 CPU 张量以节省 GPU 显存
        print(f"🎉 New best model cached in memory (Val Acc: {val_acc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            early_stop = True
            print(f"⚠️ 验证准确率连续 {PATIENCE} 轮未提升，准备早停...")

print(f"\n✅ 训练完成！最佳验证准确率: {best_val_acc:.4f}")

# =============================================
# 🔍 最终验证：加载内存中的最佳模型状态
# =============================================
if best_model_state is not None:
    # 恢复最佳权重（自动适配当前设备）
    classifier_model.load_state_dict(best_model_state)
else:
    print("⚠️ 未找到有效最佳模型，使用最终模型进行评估。")

# 在验证集上做最终评估
final_test_loss, final_test_acc = validate(classifier_model, test_loader,
                                           criterion, device)

print(f"\n🔍 最终验证结果:")
print(f"   - 测试损失 (Loss): {final_test_loss:.4f}")
print(f"   - 测试准确率 (Accuracy): {final_test_acc:.4f}")

🔍 实验 A（预训练+冻结）可训练参数量: 67,074 (约 67.1K)
Epoch 1/100 | Train Loss: 0.5452, Acc: 0.7960 | Val Loss: 0.4674, Acc: 0.8000
🎉 New best model cached in memory (Val Acc: 0.8000)
Epoch 2/100 | Train Loss: 0.4690, Acc: 0.8000 | Val Loss: 0.4480, Acc: 0.8000
Epoch 3/100 | Train Loss: 0.4174, Acc: 0.8000 | Val Loss: 0.3896, Acc: 0.8000
Epoch 4/100 | Train Loss: 0.3609, Acc: 0.8120 | Val Loss: 0.3317, Acc: 0.8100
🎉 New best model cached in memory (Val Acc: 0.8100)
Epoch 5/100 | Train Loss: 0.2985, Acc: 0.8480 | Val Loss: 0.2711, Acc: 0.8520
🎉 New best model cached in memory (Val Acc: 0.8520)
Epoch 6/100 | Train Loss: 0.2392, Acc: 0.8960 | Val Loss: 0.2056, Acc: 0.9300
🎉 New best model cached in memory (Val Acc: 0.9300)
Epoch 7/100 | Train Loss: 0.1781, Acc: 0.9540 | Val Loss: 0.1570, Acc: 0.9420
🎉 New best model cached in memory (Val Acc: 0.9420)
Epoch 8/100 | Train Loss: 0.1383, Acc: 0.9620 | Val Loss: 0.1190, Acc: 0.9700
🎉 New best model cached in memory (Val Acc: 0.9700)
Epoch 9/100 | Train Loss:

In [102]:
# ==============================
# 实验 B：无预训练（随机初始化）
# ==============================

from re import T


print("\n" + "=" * 60)
print("🧪 对照实验：随机初始化编码器（无预训练）")
print("=" * 60)

# 1. 创建全新模型（不加载任何预训练权重）
random_encoder = BlockMaskedEncoder(signal_length=SIGNAL_LENGTH)  # 随机初始化
classifier_random = MAEClassifier(
    encoder=random_encoder,
    freeze_encoder=False # 必须设为 False，否则无法训练
).to(device)

# 2. 优化器 & 学习率调度
lr_for_random = 1e-4
optimizer_for_random = torch.optim.AdamW(classifier_random.parameters(),
                                         lr=lr_for_random,
                                         weight_decay=weight_decay)
scheduler_for_random = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_for_random, T_max=num_epochs)

# 3. 早停 & 内存缓存
best_val_acc_random = 0.0
epochs_no_improve_random = 0
early_stop_flag_random = False
best_state_dict_random = None


# 打印可训练参数数量
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


num_trainable = count_trainable_params(classifier_random)
print(f"🔍 实验 B（随机+冻结）可训练参数量: {num_trainable:,} (约 {num_trainable/1e3:.1f}K)")
# 4. 训练循环
for epoch_idx in range(num_epochs):
    if early_stop_flag_random:
        print("🛑 随机模型：触发早停！训练提前结束。")
        break

    train_loss_rand, train_acc_rand = train_epoch(classifier_random,
                                                  train_loader, criterion,
                                                  optimizer_for_random, device)
    val_loss_rand, val_acc_rand = validate(classifier_random, val_loader,
                                           criterion, device)
    scheduler_for_random.step()

    print(f"[Random] Epoch {epoch_idx+1}/{num_epochs} | "
          f"Train Loss: {train_loss_rand:.4f}, Acc: {train_acc_rand:.4f} | "
          f"Val Loss: {val_loss_rand:.4f}, Acc: {val_acc_rand:.4f}")

    if val_acc_rand > best_val_acc_random + EARLY_STOP_DELTA:
        best_val_acc_random = val_acc_rand
        epochs_no_improve_random = 0
        best_state_dict_random = {
            key: tensor.cpu()
            for key, tensor in classifier_random.state_dict().items()
        }
        print(
            f"🎉 [Random] New best model cached in memory (Val Acc: {val_acc_rand:.4f})"
        )
    else:
        epochs_no_improve_random += 1
        if epochs_no_improve_random >= PATIENCE:
            early_stop_flag_random = True
            print(f"⚠️ [Random] 连续 {PATIENCE} 轮未提升，早停...")

print(f"\n✅ 随机初始化模型训练完成！最佳验证准确率: {best_val_acc_random:.4f}")

# =============================================
# 🔍 最终在测试集上评估（使用实验 B 的模型）
# =============================================
if best_state_dict_random is not None:
    classifier_random.load_state_dict(best_state_dict_random)

# 注意：这里用的是 test_loader，不是 val_loader！
final_test_loss_random, final_test_acc_random = validate(
    classifier_random, test_loader, criterion, device)

print(f"\n🔍 随机初始化模型最终测试结果:")
print(f"   - 测试损失 (Loss): {final_test_loss_random:.4f}")
print(f"   - 测试准确率 (Accuracy): {final_test_acc_random:.4f}")


🧪 对照实验：随机初始化编码器（无预训练）
🔍 实验 B（随机+冻结）可训练参数量: 1,161,410 (约 1161.4K)
[Random] Epoch 1/100 | Train Loss: 0.6596, Acc: 0.6760 | Val Loss: 0.6040, Acc: 0.8160
🎉 [Random] New best model cached in memory (Val Acc: 0.8160)
[Random] Epoch 2/100 | Train Loss: 0.5816, Acc: 0.8120 | Val Loss: 0.5419, Acc: 0.8000
[Random] Epoch 3/100 | Train Loss: 0.5297, Acc: 0.8000 | Val Loss: 0.5043, Acc: 0.8000
[Random] Epoch 4/100 | Train Loss: 0.5000, Acc: 0.8000 | Val Loss: 0.4829, Acc: 0.8000
[Random] Epoch 5/100 | Train Loss: 0.4811, Acc: 0.8000 | Val Loss: 0.4710, Acc: 0.8000
[Random] Epoch 6/100 | Train Loss: 0.4698, Acc: 0.8000 | Val Loss: 0.4631, Acc: 0.8000
[Random] Epoch 7/100 | Train Loss: 0.4594, Acc: 0.8000 | Val Loss: 0.4562, Acc: 0.8000
[Random] Epoch 8/100 | Train Loss: 0.4483, Acc: 0.8000 | Val Loss: 0.4490, Acc: 0.8000
[Random] Epoch 9/100 | Train Loss: 0.4415, Acc: 0.8000 | Val Loss: 0.4424, Acc: 0.8000
[Random] Epoch 10/100 | Train Loss: 0.4313, Acc: 0.8000 | Val Loss: 0.4351, Acc: 0.8000
[R

In [109]:
# ==============================
# 实验 C：纯线性分类器（无编码器）
# ==============================

print("\n" + "=" * 60)
print("🧪 对照实验：纯线性分类器（Linear Only）")
print("=" * 60)

# 获取信号长度 L（从数据中推断）
sample_input, _ = next(iter(train_loader))
L = sample_input.shape[-1]  # 假设 x: [B, 1, L]


class LinearOnlyClassifier(nn.Module):

    def __init__(self, input_len, num_classes):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(input_len, num_classes)

    def forward(self, x):
        # x: [B, 1, L] → [B, L]
        x = self.flatten(x)  # [B, L]
        return self.linear(x)


# 创建模型
linear_model = LinearOnlyClassifier(input_len=L,
                                    num_classes=num_classes).to(device)

# 优化器（线性模型可稍大学习率）
lr_linear = 1e-3
optimizer_linear = torch.optim.AdamW(linear_model.parameters(),
                                     lr=lr_linear,
                                     weight_decay=1e-4)
scheduler_linear = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_linear,
                                                              T_max=num_epochs)

# 训练循环（复用已有函数）
best_val_acc_lin = 0.0
epochs_no_improve_lin = 0
early_stop_lin = False

for epoch in range(num_epochs):
    if early_stop_lin:
        print("🛑 线性模型：触发早停！训练提前结束。")
        break

    train_loss, train_acc = train_epoch(linear_model, train_loader, criterion,
                                        optimizer_linear, device)
    val_loss, val_acc = validate(linear_model, val_loader, criterion, device)
    scheduler_linear.step()

    print(f"[Linear] Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    if val_acc > best_val_acc_lin + EARLY_STOP_DELTA:
        best_val_acc_lin = val_acc
        epochs_no_improve_lin = 0
        torch.save(linear_model.state_dict(), "best_linear_classifier.pth")
        print(f"🎉 [Linear] New best model saved (Val Acc: {val_acc:.4f})")
    else:
        epochs_no_improve_lin += 1
        if epochs_no_improve_lin >= PATIENCE:
            early_stop_lin = True
            print(f"⚠️ [Linear] 连续 {PATIENCE} 轮未提升，早停...")

print(f"\n✅ 线性分类器训练完成！最佳验证准确率: {best_val_acc_lin:.4f}")


🧪 对照实验：纯线性分类器（Linear Only）
[Linear] Epoch 1/100 | Train Loss: 1.6900, Acc: 0.2760 | Val Loss: 1.7778, Acc: 0.2700
🎉 [Linear] New best model saved (Val Acc: 0.2700)
[Linear] Epoch 2/100 | Train Loss: 1.2956, Acc: 0.5380 | Val Loss: 1.8520, Acc: 0.2880
🎉 [Linear] New best model saved (Val Acc: 0.2880)
[Linear] Epoch 3/100 | Train Loss: 1.1135, Acc: 0.6380 | Val Loss: 1.9082, Acc: 0.3140
🎉 [Linear] New best model saved (Val Acc: 0.3140)
[Linear] Epoch 4/100 | Train Loss: 0.9834, Acc: 0.6940 | Val Loss: 1.9621, Acc: 0.3060
[Linear] Epoch 5/100 | Train Loss: 0.8834, Acc: 0.7580 | Val Loss: 2.0125, Acc: 0.3220
🎉 [Linear] New best model saved (Val Acc: 0.3220)
[Linear] Epoch 6/100 | Train Loss: 0.8073, Acc: 0.7880 | Val Loss: 2.0476, Acc: 0.3120
[Linear] Epoch 7/100 | Train Loss: 0.7372, Acc: 0.8260 | Val Loss: 2.0868, Acc: 0.3240
🎉 [Linear] New best model saved (Val Acc: 0.3240)
[Linear] Epoch 8/100 | Train Loss: 0.6851, Acc: 0.8440 | Val Loss: 2.1260, Acc: 0.3140
[Linear] Epoch 9/100 | Tra


📊 最终性能对比
方法                        | 最佳验证准确率
----------------------------------------------------------------------
MAE 预训练 + 微调              | 0.9980


NameError: name 'best_val_acc_rand' is not defined